In [21]:
from hftbacktest.data.utils import tardis, binancefutures
import numpy as np
import polars as pl
import pandas as pd
from hftbacktest import BacktestAsset, HashMapMarketDepthBacktest
from numba import njit


In [2]:
data = tardis.convert(
    ['../exampleData/BTCUSDT_trades.csv.gz', '../exampleData/BTCUSDT_book.csv.gz']
)

Reading ../exampleData/BTCUSDT_trades.csv.gz
Reading ../exampleData/BTCUSDT_book.csv.gz
Correcting the latency
Correcting the event order


In [3]:
pl.DataFrame(data)

ev,exch_ts,local_ts,px,qty,order_id,ival,fval
u64,i64,i64,f64,f64,u64,i64,f64
3758096386,1580515202342000000,1580515202497052000,9364.51,1.197,0,0,0.0
3758096386,1580515202342000000,1580515202497346000,9365.67,0.02,0,0,0.0
3758096386,1580515202342000000,1580515202497352000,9365.86,0.01,0,0,0.0
3758096386,1580515202342000000,1580515202497357000,9366.36,0.002,0,0,0.0
3758096386,1580515202342000000,1580515202497363000,9366.36,0.003,0,0,0.0
…,…,…,…,…,…,…,…
3489660929,1580601599823000000,1580601599944404000,9397.79,0.0,0,0,0.0
3758096385,1580601599833000000,1580601599952176000,9354.8,4.07,0,0,0.0
3758096385,1580601599842000000,1580601599962961000,9351.47,3.914,0,0,0.0


In [4]:
_ = tardis.convert(
    ['../exampleData/BTCUSDT_trades.csv.gz', '../exampleData/BTCUSDT_book.csv.gz'],
    output_filename='../exampleData/btcusdt_20200201.npz',
    buffer_size=200_000_000
)

Reading ../exampleData/BTCUSDT_trades.csv.gz
Reading ../exampleData/BTCUSDT_book.csv.gz
Correcting the latency
Correcting the event order
Saving to ../exampleData/btcusdt_20200201.npz


In [2]:
# numba.njit is strongly recommended for fast backtesting.
@njit
def print_bbo(hbt):
    # Iterating until hftbacktest reaches the end of data.
    # Elapses 60-sec every iteration.
    # Time unit is the same as data's timestamp's unit.
    # Timestamp of the sample data is in nanoseconds.
    while hbt.elapse(60 * 1e9) == 0:        
        # Gets the market depth for the first asset.
        depth = hbt.depth(0)

        # Prints the best bid and the best offer.
        print(
            'current_timestamp:', hbt.current_timestamp,
            ', best_bid:', np.round(depth.best_bid, 1),
            ', best_ask:', np.round(depth.best_ask, 1)
        )
    return True

In [33]:
asset = (
    BacktestAsset()
        # Sets the data to feed for this asset.
        #
        # Due to the vast size of tick-by-tick market depth and trade data,
        # loading the entire dataset into memory can be challenging,
        # particularly when backtesting across multiple days.
        # HftBacktest offers lazy loading support and is compatible with npy and preferably npz.
        #
        # For details on the normalized feed data, refer to the following documents.
        # * https://hftbacktest.readthedocs.io/en/latest/data.html    
        # * https://hftbacktest.readthedocs.io/en/latest/tutorials/Data%20Preparation.html
        .data(['exampleData/btcusdt_20200201.npz'])
        # Sets the initial snapshot (optional).
        # .initial_snapshot('usdm/btcusdt_20240808_eod.npz')
        # Asset type:
        # * Linear
        # * Inverse.
        # 1.0 represents the contract size, which is the value of the asset per quoted price.
        .linear_asset(1.0) 
        # HftBacktest provides two built-in latency models.
        # * constant_latency
        # * intp_order_latency
        # To implement your own latency model, please use Rust.
        # 
        # Time unit is the same as data's timestamp's unit. Timestamp of the sample data is in nanoseconds.
        # Sets the order entry latency and response latency to 10ms.
        .constant_latency(10_000_000, 10_000_000)
        # HftBacktest provides several types of built-in queue position models.
        # Please find the details in the documents below.
        # https://hftbacktest.readthedocs.io/en/latest/tutorials/Probability%20Queue%20Models.html
        #
        # To implement your own queue position model, please use Rust.
        .risk_adverse_queue_model() 
        # HftBacktest provides two built-in exchange models.
        # * no_partial_fill_exchange
        # * partial_fill_exchange
        # To implement your own exchange model, please use Rust.
        .no_partial_fill_exchange()
        # HftBacktest provides several built-in fee models.
        # * trading_value_fee_model
        # * trading_qty_fee_model
        # * flat_per_trade_fee_model
        #
        # 0.02% maker fee and 0.07% taker fee. If the fee is negative, it represents a rebate.
        # For example, -0.00005 represents a 0.005% rebate for the maker order.
        .trading_value_fee_model(0.0002, 0.0007)
        # Tick size of this asset: minimum price increasement
        .tick_size(0.1)
        # Lot size of this asset: minimum trading unit.
        .lot_size(0.001)
        # Sets the capacity of the vector that stores trades occurring in the market.
        # If you set the size, you need call `clear_last_trades` to clear the vector.
        # A value of 0 indicates that no market trades are stored. (Default)
        .last_trades_capacity(0)
)

# HftBacktest provides several types of built-in market depth implementations.
# HashMapMarketDepthBacktest constructs a Backtest using a HashMap-based market depth implementation.
# Another useful implementation is ROIVectorMarketDepth, which is utilized in ROIVectorMarketDepthBacktest.
# Please find the details in the document below.
hbt = HashMapMarketDepthBacktest([asset])

/var/folders/1h/62kcxqln6g32kv_25b34rd6h0000gn/T/ipykernel_77882/2712221128.py:28: DeprecationWarning: constant_latency() is deprecated; use constant_order_latency().
  .constant_latency(10_000_000, 10_000_000)


In [34]:
print_bbo(hbt)

current_timestamp: 1580515262342000000 , best_bid: 9372.4 , best_ask: 9374.4
current_timestamp: 1580515322342000000 , best_bid: 9372.4 , best_ask: 9373.5
current_timestamp: 1580515382342000000 , best_bid: 9369.5 , best_ask: 9369.9
current_timestamp: 1580515442342000000 , best_bid: 9365.8 , best_ask: 9367.1
current_timestamp: 1580515502342000000 , best_bid: 9361.2 , best_ask: 9361.7
current_timestamp: 1580515562342000000 , best_bid: 9363.9 , best_ask: 9365.2
current_timestamp: 1580515622342000000 , best_bid: 9365.1 , best_ask: 9365.8
current_timestamp: 1580515682342000000 , best_bid: 9364.4 , best_ask: 9365.3
current_timestamp: 1580515742342000000 , best_bid: 9366.6 , best_ask: 9366.7
current_timestamp: 1580515802342000000 , best_bid: 9368.6 , best_ask: 9368.7
current_timestamp: 1580515862342000000 , best_bid: 9385.4 , best_ask: 9386.2
current_timestamp: 1580515922342000000 , best_bid: 9379.1 , best_ask: 9379.5
current_timestamp: 1580515982342000000 , best_bid: 9384.3 , best_ask: 9386.0

True

In [35]:
_ = hbt.close()